# Mask Ablation Experiment — LCLD

Runs mask variants M2 (directional), M3 (derived-feature freeze), M4 (term OHE freeze), M5 (combined), M6-strict, M6-relaxed on LCLD with the neural model and CAPGD attack. Loads M0/M1 baselines from existing comparison results. Produces summary tables, feasibility audit, perturbation stats, and E1 cost-weighted analysis.

Reference: `notebooks/tabularbench_comparison.ipynb` (reused setup + utility functions).
Spec: `docs/plans/mask_ablation_experiment_plan.md`.

In [ ]:
# Cell 1: Verify GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/FraudBench"
for subdir in ["data", "results", "results/adv_examples"]:
    os.makedirs(os.path.join(DRIVE_ROOT, subdir), exist_ok=True)
print("Google Drive mounted.")

In [ ]:
# Cell 3: Clone or update repo
import os, shutil

REPO_URL = "https://github.com/iHaydenzZ/Capstone_FraudBench.git"
REPO_DIR = "/content/Capstone_FraudBench"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    os.chdir(REPO_DIR)
    !git pull
else:
    os.chdir("/content")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")
!git log --oneline -3

In [ ]:
# Cell 4: Install dependencies
# Colab's pre-installed numpy/scipy can conflict with project deps.
# Force a compatible set, then restart the runtime so the C extensions reload.
!pip install "numpy<2.1" "scipy>=1.14,<1.15" "scikit-learn>=1.5" -q 2>&1 | tail -5
!pip install -e . --no-deps -q 2>&1 | tail -5
!pip install "numba>=0.61" -q 2>&1 | tail -3
!pip install xgboost torch art pyyaml joblib pandas -q 2>&1 | tail -3

# --- IMPORTANT ---
# After this cell finishes, restart the runtime:
#   Runtime > Restart session  (or Ctrl+M then .)
# Then skip this cell and continue from Cell 5.
# The restart is needed because Colab caches numpy's C extensions in memory.
print("\n>>> RESTART THE RUNTIME NOW, then skip this cell and run from Cell 5. <<<")

In [ ]:
# Cell 5: Symlink datasets from Google Drive
import os

DRIVE_DATA = "/content/drive/MyDrive/FraudBench/data"
DATASETS_DIR = "/content/Capstone_FraudBench/datasets"

for dataset_dir in ["CCFD", "ieee-fraud-detection", "LCLD", "Sparkov"]:
    src = os.path.join(DRIVE_DATA, dataset_dir)
    dst = os.path.join(DATASETS_DIR, dataset_dir)
    if os.path.islink(dst):
        os.unlink(dst)
    if os.path.exists(src):
        os.symlink(src, dst)
        print(f"  Linked: {dataset_dir}/")
    else:
        print(f"  NOT FOUND: {dataset_dir}/ -- upload to {src}")

print("Dataset symlinks ready.")

In [ ]:
# Cell 6: Mask variant configuration
#
# M3/M4/M5 are defined by their IMMUTABLE raw-feature sets (extends LCLD_IMMUTABLE_RAW).
# M6-strict and M6-relaxed are defined by their MUTABLE raw-feature sets; the experiment
# loop (Cell 8) inverts them against dataset.X.columns to get the immutable set to
# pass into build_processed_mutable_mask().

from typing import Set

# --- Baseline immutable set (copied verbatim from tabularbench_comparison.ipynb Cell 6) ---
LCLD_IMMUTABLE_RAW: Set[str] = {
    # LC internal pricing/grading
    "grade", "sub_grade", "int_rate", "installment",
    "funded_amnt", "funded_amnt_inv", "initial_list_status",
    # LC verification outcomes
    "verification_status", "verification_status_joint",
    # Credit bureau data
    "delinq_2yrs", "inq_last_6mths", "inq_last_12m", "inq_fi",
    "open_acc", "open_acc_6m", "open_act_il",
    "open_il_12m", "open_il_24m", "open_rv_12m", "open_rv_24m",
    "pub_rec", "pub_rec_bankruptcies", "total_acc",
    "revol_bal", "revol_util", "il_util", "all_util",
    "tot_cur_bal", "tot_hi_cred_lim", "total_bal_il",
    "total_rev_hi_lim", "max_bal_bc",
    "pct_tl_nvr_dlq", "percent_bc_gt_75",
    "collections_12_mths_ex_med",
    "mths_since_last_delinq", "mths_since_last_il_delinq",
    "mths_since_last_major_delinq", "mths_since_last_record",
    "mths_since_rcnt_il",
    "payment_inc_ratio",
}

LCLD_MUTABLE_RAW: Set[str] = {
    "loan_amnt", "term", "purpose", "emp_length",
    "annual_inc", "annual_inc_joint", "home_ownership",
    "dti", "dti_joint", "application_type", "addr_state",
}

# --- Variant immutable sets ---
LCLD_IMMUTABLE_M3 = LCLD_IMMUTABLE_RAW | {"dti", "dti_joint"}
LCLD_IMMUTABLE_M4 = LCLD_IMMUTABLE_RAW | {"term"}
LCLD_IMMUTABLE_M5 = LCLD_IMMUTABLE_RAW | {"dti", "dti_joint", "term"}

# --- M6 attacker-capability profiles ---
MUTABLE_STRICT: Set[str] = {
    "loan_amnt", "purpose", "home_ownership", "application_type", "addr_state",
}
MUTABLE_RELAXED: Set[str] = {
    "loan_amnt", "purpose", "home_ownership", "application_type", "addr_state",
    "annual_inc", "annual_inc_joint", "emp_length",
}

# --- M2 directionality ---
# Only emp_length is increase-only among LCLD mutable features.
DIRECTION_CONSTRAINTS = {"emp_length": "increase"}

# --- E1 cost weights (raw-space, normalized units) ---
FEATURE_COSTS = {
    "loan_amnt":        1.0,
    "purpose":          0.5,
    "home_ownership":   3.0,
    "addr_state":       2.0,
    "application_type": 1.0,
    "annual_inc":       8.0,
    "annual_inc_joint": 8.0,
    "emp_length":       5.0,
    "dti":              7.0,
    "dti_joint":        7.0,
    "term":             1.0,
}

# --- Sanity checks ---
assert LCLD_IMMUTABLE_RAW.isdisjoint(LCLD_MUTABLE_RAW), "raw sets overlap"
assert set(FEATURE_COSTS.keys()) == LCLD_MUTABLE_RAW, "cost dict must cover all mutable features"
assert MUTABLE_STRICT <= LCLD_MUTABLE_RAW, "strict profile contains non-mutable feature"
assert MUTABLE_RELAXED <= LCLD_MUTABLE_RAW, "relaxed profile contains non-mutable feature"
assert LCLD_IMMUTABLE_M5 == LCLD_IMMUTABLE_M3 | LCLD_IMMUTABLE_M4, "M5 must equal M3 ∪ M4"
assert all(v in {"increase", "decrease"} for v in DIRECTION_CONSTRAINTS.values()), "unknown direction constraint"

print(f"Baseline mutable:   {len(LCLD_MUTABLE_RAW)} raw features")
print(f"M3 locks adds:      {sorted(LCLD_IMMUTABLE_M3 - LCLD_IMMUTABLE_RAW)}")
print(f"M4 locks adds:      {sorted(LCLD_IMMUTABLE_M4 - LCLD_IMMUTABLE_RAW)}")
print(f"M5 locks adds:      {sorted(LCLD_IMMUTABLE_M5 - LCLD_IMMUTABLE_RAW)}")
print(f"M6-strict mutable:  {sorted(MUTABLE_STRICT)} ({len(MUTABLE_STRICT)} features)")
print(f"M6-relaxed mutable: {sorted(MUTABLE_RELAXED)} ({len(MUTABLE_RELAXED)} features)")